In [1]:
# === INSTALACIÓN DE DEPENDENCIAS (correr solo la primera vez) ===
 

!pip install openai anthropic chromadb ipywidgets --quiet

In [1]:
!pip install chromadb

In [2]:
import os, json
from getpass import getpass

# === 🔑 API KEYS (pega tus claves al ejecutar) ===
os.environ["OPENAI_API_KEY"]    = getpass("OPENAI_API_KEY    ➜ ")   # embeddings
os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY ➜ ")   # LLM

# === 📚 CARGA DE LA BASE DE CONOCIMIENTO ===
with open("knowledge_base.json", "r", encoding="utf-8") as f:
    kb = json.load(f)
n = len(kb["enfermedades"])

# === 🧩 CHUNKS: 1 por enfermedad (síntomas + descripción → texto a embebir) ===
chunks = [{
    "text": f"Enfermedad: {kb['enfermedades'][i]}. Síntomas: {kb['sintomas'][i]}. Descripción: {kb['descripciones'][i]}",
    "metadata": {
        "enfermedad":  kb["enfermedades"][i],
        "tratamiento": kb["tratamientos"][i],
        "advertencia": kb["advertencias"][i],
    }
} for i in range(n)]

print(f"✅ {n} entradas clínicas cargadas y chunkeadas")

OPENAI_API_KEY    ➜  ········
ANTHROPIC_API_KEY ➜  ········


✅ 14 entradas clínicas cargadas y chunkeadas


In [3]:
import chromadb
from openai import OpenAI
from chromadb import EmbeddingFunction, Documents, Embeddings

# === 🟢 CLIENTE OPENAI (embeddings) ===
openai_client = OpenAI()
EMBEDDING_MODEL = "text-embedding-3-small"   # 1536 dims, rápido y económico

class OpenAIEmbedder(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        r = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=list(input))
        return [d.embedding for d in r.data]
    def name(self): return EMBEDDING_MODEL

# === 🗄️ CHROMADB PERSISTENTE (cosine similarity) ===
chroma = chromadb.PersistentClient(path="./chroma_clinical")
vector_store = chroma.get_or_create_collection(
    name="clinical_kb",
    embedding_function=OpenAIEmbedder(),
    metadata={"hnsw:space": "cosine"},
)

# === 📥 INDEXADO (solo si está vacía → ahorra llamadas a OpenAI) ===
if vector_store.count() == 0:
    vector_store.add(
        documents=[c["text"] for c in chunks],
        metadatas=[c["metadata"] for c in chunks],
        ids=[f"clin_{i}" for i in range(len(chunks))],
    )
    print(f"📥 Indexados {vector_store.count()} chunks")
else:
    print(f"♻️  Reutilizando colección con {vector_store.count()} chunks")

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_17660\2002793438.py:19: DeprecationWarning: The class OpenAIEmbedder does not implement __init__. This will be required in a future version.
  embedding_function=OpenAIEmbedder(),


📥 Indexados 14 chunks


In [4]:
from anthropic import Anthropic

# === 🟣 CLIENTE CLAUDE (LLM generador) ===
claude = Anthropic()
LLM_MODEL = "claude-sonnet-4-5"

# === 📜 SYSTEM PROMPT (rol restrictivo → evita alucinaciones) ===
SYSTEM_PROMPT = """Eres un asistente clínico educativo. Tu única fuente son los fragmentos entregados.

Reglas:
- Responde en español, tono claro y empático.
- Estructura: 🩺 Posible condición · 📋 Descripción · 💊 Tratamiento · ⚠️ Advertencia.
- Si no hay coincidencia, responde: "No encontré coincidencias en la base de conocimiento."
- NUNCA inventes diagnósticos ni tratamientos.
- SIEMPRE recuerda que esto NO sustituye una consulta médica."""

# === 🔍 RETRIEVAL: top-K chunks por similitud semántica ===
def retrieve_clinical(consulta: str, k: int = 3):
    r = vector_store.query(query_texts=[consulta], n_results=k)
    return [
        {"text": d, "metadata": m, "distance": dist}
        for d, m, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])
    ]

# === 🤖 GENERACIÓN: contexto recuperado → Claude redacta ===
def generate_clinical_answer(consulta: str, k: int = 3):
    retrieved = retrieve_clinical(consulta, k)
    contexto = "\n\n---\n\n".join(
        f"[Fragmento {i+1}]\nEnfermedad: {r['metadata']['enfermedad']}\n"
        f"Texto: {r['text']}\nTratamiento: {r['metadata']['tratamiento']}\n"
        f"Advertencia: {r['metadata']['advertencia']}"
        for i, r in enumerate(retrieved)
    )
    response = claude.messages.create(
        model=LLM_MODEL,
        max_tokens=800,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": f"CONTEXTO:\n{contexto}\n\nCONSULTA:\n{consulta}"}],
    )
    return {"answer": response.content[0].text, "sources": retrieved}

# === 🧪 PRUEBA RÁPIDA ===
demo = generate_clinical_answer("Tengo picor en la piel y se ve seca y agrietada", k=3)
print(demo["answer"])
print("\n📚 FUENTES:")
for i, s in enumerate(demo["sources"], 1):
    print(f"   {i}. dist={s['distance']:.4f} → {s['metadata']['enfermedad']}")

🩺 **Posible condición**

Basado en tus síntomas (picor, piel seca y agrietada), podrías estar presentando **Piel seca (xerosis cutis)** o **Picor crónico (prurito)** asociado a sequedad cutánea.

---

📋 **Descripción**

La **xerosis cutis** es una sequedad extrema de la piel que afecta a millones de personas. Puede manifestarse con piel escamosa, agrietada y picor. En algunos casos, puede estar asociada a enfermedades dermatológicas, internas o neurológicas.

El **prurito** puede volverse crónico si persiste más de 6 semanas, afectando significativamente la calidad de vida.

---

💊 **Tratamiento sugerido**

- **Hidratación cutánea** con cremas emolientes varias veces al día
- **Evitar jabones agresivos** y usar productos suaves para piel sensible
- **Antihistamínicos** si el picor es muy intenso
- **Evitar rascarse** para no empeorar las lesiones
- Tratar cualquier enfermedad de base que pueda estar causando la sequedad

---

⚠️ **ADVERTENCIA IMPORTANTE**

- La piel seca puede ser sínt

In [5]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# === 🎨 ENCABEZADO ===
header = widgets.HTML("""
<div style='background: linear-gradient(135deg, #0ea5e9 0%, #6366f1 100%);
            padding: 22px; border-radius: 14px; color: white;
            font-family: -apple-system, "Segoe UI", sans-serif; margin-bottom: 14px;'>
    <h2 style='margin:0;'>🩺 Asistente Clínico RAG</h2>
    <p style='margin:6px 0 0 0; opacity:0.92; font-size:0.95em;'>
        Síntomas → Posible condición · Descripción · Tratamiento · Advertencia
    </p>
    <p style='margin:6px 0 0 0; opacity:0.85; font-size:0.82em;'>
        ⚠️ Demostración educativa — NO sustituye consulta médica.
    </p>
</div>""")

# === 🧰 CONTROLES ===
input_query = widgets.Textarea(
    placeholder="Ej: Tengo dolor pélvico fuerte durante la menstruación...",
    layout=widgets.Layout(width="100%", height="90px"), description="🩹")
slider_k = widgets.IntSlider(value=3, min=1, max=5, description="Coincidencias:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="55%"))
btn_ask = widgets.Button(description="🔍 Analizar", button_style="primary",
    layout=widgets.Layout(width="180px", height="42px"))
btn_clear = widgets.Button(description="🧹 Limpiar",
    layout=widgets.Layout(width="120px", height="42px"))
panel = widgets.Output()

# === 🖼️ RENDER DE RESPUESTA ===
def render(query, payload):
    sources_html = "".join(f"""
        <details style='margin:8px 0;padding:10px;background:#f1f5f9;
                        border-left:3px solid #6366f1;border-radius:6px;'>
            <summary style='cursor:pointer;font-weight:600;color:#475569;'>
                📄 {s['metadata']['enfermedad']} — distancia: {s['distance']:.4f}
            </summary>
            <p style='margin-top:8px;color:#1e293b;font-size:0.9em;line-height:1.55;'>
                <b>Síntomas:</b> {s['text']}<br><br>
                <b>💊 Tratamiento:</b> {s['metadata']['tratamiento']}<br><br>
                <b>⚠️ Advertencia:</b> {s['metadata']['advertencia']}
            </p>
        </details>""" for s in payload["sources"])

    html = f"""
    <div style='font-family:-apple-system,"Segoe UI",sans-serif;max-width:920px;'>
        <div style='background:#fff;padding:18px;border-radius:10px;
                    border:1px solid #e2e8f0;margin-bottom:14px;'>
            <div style='color:#0ea5e9;font-weight:600;font-size:0.82em;
                        text-transform:uppercase;'>Consulta</div>
            <div style='font-size:1.05em;color:#0f172a;margin-top:6px;'>{query}</div>
        </div>
        <div style='background:#fff;padding:20px;border-radius:10px;
                    border:1px solid #e2e8f0;margin-bottom:14px;'>
            <div style='color:#10b981;font-weight:600;font-size:0.82em;
                        text-transform:uppercase;'>💡 Análisis de Claude</div>
            <div style='font-size:1.02em;color:#0f172a;margin-top:10px;
                        line-height:1.7;white-space:pre-wrap;'>{payload["answer"]}</div>
        </div>
        <div style='background:#fff;padding:16px;border-radius:10px;border:1px solid #e2e8f0;'>
            <div style='color:#64748b;font-weight:600;font-size:0.82em;
                        text-transform:uppercase;margin-bottom:8px;'>🔍 Coincidencias</div>
            {sources_html}
        </div>
    </div>"""
    display(HTML(html))

# === 🎯 EVENTOS ===
def on_ask(_):
    q = input_query.value.strip()
    if not q:
        with panel: clear_output(); print("⚠️ Escribe una consulta primero.")
        return
    with panel:
        clear_output(); print("⏳ Buscando coincidencias y consultando a Claude...")
        result = generate_clinical_answer(q, k=slider_k.value)
        clear_output(); render(q, result)

def on_clear(_):
    input_query.value = ""
    with panel: clear_output()

btn_ask.on_click(on_ask)
btn_clear.on_click(on_clear)

# === 🚀 LANZAR UI ===
display(widgets.VBox([header, input_query,
    widgets.HBox([slider_k]), widgets.HBox([btn_ask, btn_clear]), panel]))